<a href="https://colab.research.google.com/github/pinkprincess536/eeg/blob/main/final_train_eeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

rebelxhearts_final_eeg_all_patients_path = kagglehub.dataset_download('rebelxhearts/final-eeg-all-patients')
rebelxhearts_new_final_path = kagglehub.dataset_download('rebelxhearts/new-final')

print('Data source import complete.')


<a href="https://colab.research.google.com/github/pinkprincess536/eeg/blob/main/train_eeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

rebelxhearts_npy_traintestdata_path = kagglehub.dataset_download('rebelxhearts/final-eeg-all-patients')

print('Data source import complete.')


Data source import complete.


In [ ]:
!pip install torch numpy scikit-learn matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.is_available()}")

PyTorch 2.10.0+cpu | GPU: False


## Cell 1: Config

All tunable training parameters in one place.
Data path points to Kaggle Dataset (no Drive, no MNE, no EDF).

In [ ]:
!pip install -q mlflow
import mlflow
import os

mlflow.set_experiment("seizure-weight-comparison")

WEIGHTS_TO_TEST = [ 20.0]
print(f"Will test seizure weights: {WEIGHTS_TO_TEST}")
print(f"Experiment: {mlflow.get_experiment_by_name('seizure-weight-comparison').experiment_id}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 88.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 72.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.9/936.9 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 10.2 MB/s eta 0:00:00


2026/06/12 08:15:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/12 08:15:21 INFO mlflow.store.db.utils: Updating database tables
2026/06/12 08:15:24 INFO mlflow.tracking.fluent: Experiment with name 'seizure-weight-comparison' does not exist. Creating a new experiment.


Will test seizure weights: [20.0]
Experiment: 1


In [ ]:
# --- PATHS ---
DATA_DIR = "/kaggle/input/datasets/rebelxhearts/new-final/processed"

# --- TRAINING ---
BATCH_SIZE      = 8
LEARNING_RATE   = 0.001
NUM_EPOCHS      = 13
RANDOM_SEED     = 42
     # missed seizures cost 7x more -> high recall

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Data: {DATA_DIR}")
print(f"Batch: {BATCH_SIZE} | LR: {LEARNING_RATE} | Epochs: {NUM_EPOCHS}")

print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Data: /kaggle/input/datasets/rebelxhearts/new-final/processed
Batch: 8 | LR: 0.001 | Epochs: 13
GPU: False


## Cell 1b: DVC Pull (Optional)

If your dataset is versioned with DVC, this pulls the latest version from Google Drive. Skips download if already up to date.

## Cell 2: Load Preprocessed Data

Loads .npy files saved by the Colab preprocessing notebook.
No EDF files. No MNE. No filtering. Just numpy arrays.

In [ ]:
import os

print("Loading preprocessed data...")

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
X_test  = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_test  = np.load(os.path.join(DATA_DIR, "y_test.npy"))

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape},  y={y_test.shape}")

# Print class distribution
ut, ct = np.unique(y_train, return_counts=True)
print(f"Train classes: {dict(zip(ut, ct))}")
ut, ct = np.unique(y_test, return_counts=True)
print(f"Test classes:  {dict(zip(ut, ct))}")

# Print metadata if available
info_path = os.path.join(DATA_DIR, "info.txt")
if os.path.exists(info_path):
    with open(info_path, "r") as f:
        print("--- Metadata ---{f.read()}")
else:
    print("No info.txt found (optional metadata file)")

Loading preprocessed data...
Train: X=(889, 22, 1792), y=(889,)
Test:  X=(9129, 22, 1792),  y=(9129,)
Train classes: {np.int64(0): np.int64(500), np.int64(1): np.int64(389)}
Test classes:  {np.int64(0): np.int64(9000), np.int64(1): np.int64(129)}
--- Metadata ---{f.read()}


## Cell 3: Data Augmentation

Conservative augmentation for seizure windows only.
Time shift +-200ms, amplitude x0.85-1.15, Gaussian noise sigma=0.01.
Applied on-the-fly during training, not to non-seizure windows.

In [ ]:
def augment_seizure(windows, seed=None):
    """Augment seizure windows: time shift + scale + noise."""
    if seed is not None:
        np.random.seed(seed)
    aug = windows.copy()
    B, C, T = aug.shape
    for b in range(B):
        shift = np.random.randint(-51, 51)
        aug[b] = np.roll(aug[b], shift, axis=-1)
        aug[b] *= np.random.uniform(0.85, 1.15)
        aug[b] += np.random.randn(C, T).astype(np.float32) * 0.01
    return aug

print("Augmentation function ready.")

Augmentation function ready.


## Cell 4: PyTorch Tensors + DataLoader

1D CNN input shape: (batch, 23 channels, 1792 time samples).

In [ ]:
Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xte = torch.tensor(X_test,  dtype=torch.float32)
yte = torch.tensor(y_test,  dtype=torch.long)

train_ds = TensorDataset(Xtr, ytr)
test_ds  = TensorDataset(Xte, yte)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Input shape: {Xtr.shape}  (samples, 23 channels, 1792 time)")
print(f"Train batches: {len(train_loader)} | Test: {len(test_loader)}")

Input shape: torch.Size([889, 22, 1792])  (samples, 23 channels, 1792 time)
Train batches: 112 | Test: 1142


## Cell 5: 1D CNN Model

3 Conv1d blocks (k7, k5, k3) + BatchNorm + Dropout + AdaptiveAvgPool1d(16)

```
Block 1: Conv1d(23->32,k=7) -> BN -> ReLU -> MaxPool(4)  (1792->448)
Block 2: Conv1d(32->64,k=5) -> BN -> ReLU -> MaxPool(4)  (448->112)
Block 3: Conv1d(64->128,k=3)-> BN -> ReLU -> AdaptiveAvgPool1d(16)
Flatten -> Dropout(0.4) -> FC(2048->64) -> Dropout(0.3) -> FC(64->2)
```

In [ ]:
n_ch = X_train.shape[1]  # auto-detect channel count from data

class EEGCNN1D(nn.Module):
    def __init__(self, n_channels=n_ch, n_classes=2):
        super().__init__()
        # Block 1
        self.conv1 = nn.Conv1d(n_channels, 32, kernel_size=7, padding=3)
        self.bn1   = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(4)

        # Block 2
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2   = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(4)

        # Block 3
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm1d(128)
        self.adapt = nn.AdaptiveAvgPool1d(16)

        # Classifier
        self.drop1 = nn.Dropout(0.4)
        self.fc1   = nn.Linear(128 * 16, 64)
        self.drop2 = nn.Dropout(0.3)
        self.fc2   = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))  # (B,32,448)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))  # (B,64,112)
        x = F.relu(self.bn3(self.conv3(x)))               # (B,128,112)
        x = self.adapt(x)                                 # (B,128,16)
        x = torch.flatten(x, 1)                           # (B,2048)
        x = self.drop1(x)
        x = F.relu(self.fc1(x))                           # (B,64)
        x = self.drop2(x)
        x = self.fc2(x)                                   # (B,2)
        return x

# Move model to GPU if available

# Verify


## Cell 6: Train 1D CNN with Augmentation

Augmentation applied ONLY to seizure windows.
1 conservative copy per seizure window per epoch.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results = []

for seizure_weight in WEIGHTS_TO_TEST:
    # ---- FRESH MODEL EACH RUN ----
    model = EEGCNN1D(n_channels=n_ch).to(device)

    with mlflow.start_run(run_name=f"weight-{seizure_weight:.0f}x"):
        # ---- LOG PARAMS ----
        mlflow.log_param("seizure_weight", seizure_weight)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("lr", LEARNING_RATE)
        mlflow.log_param("n_epochs", NUM_EPOCHS)
        mlflow.log_param("n_channels", n_ch)

        # ---- CRITERION + OPTIMIZER ----
        criterion = nn.CrossEntropyLoss(
            weight=torch.tensor([1.0, seizure_weight]).to(device))
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        # ---- METRIC LISTS (reset each run) ----
        train_recall = []
        train_precision = []
        train_specificity = []
        train_accuracy = []
        train_epoch = []

        # ---- TRAINING LOOP ----
        for epoch in range(NUM_EPOCHS):
            model.train()
            running_loss = 0.0
            n_batches = 0

            for x, y in train_loader:
                x, y = x.to(device), y.to(device)

                # Augment seizure windows only
                sm = (y == 1)
                if sm.any():
                    si = x[sm].cpu().numpy()
                    sl = y[sm]
                    aug_np = augment_seizure(si)
                    aug_t = torch.tensor(aug_np, dtype=torch.float32).to(device)
                    x_aug = torch.cat([x, aug_t], dim=0)
                    y_aug = torch.cat([y, sl], dim=0)
                else:
                    x_aug, y_aug = x, y

                optimizer.zero_grad()
                outputs = model(x_aug)
                loss = criterion(outputs, y_aug)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                n_batches += 1

            avg_loss = running_loss / n_batches

            # Evaluate on TRAINING set (no test leakage, just monitoring) ---
            model.eval()
            all_preds_tr, all_true_tr = [], []
            with torch.no_grad():
                for x, y in train_loader:
                    x, y = x.to(device), y.to(device)
                    o = model(x)
                    _, p = torch.max(o, 1)
                    all_preds_tr.extend(p.cpu().numpy())
                    all_true_tr.extend(y.cpu().numpy())

            all_preds_tr = np.array(all_preds_tr)
            all_true_tr  = np.array(all_true_tr)

            # Recall = TP / (TP + FN)
            sez_mask = all_true_tr == 1
            tp = (all_preds_tr[sez_mask] == 1).sum()
            fn = (all_preds_tr[sez_mask] == 0).sum()
            rec = tp / (tp + fn) if (tp + fn) > 0 else 0

            # Precision = TP / (TP + FP)
            pred_sez = all_preds_tr == 1
            fp = (all_true_tr[pred_sez] == 0).sum()
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0

            # Specificity = TN / (TN + FP)
            non_mask = all_true_tr == 0
            tn = (all_preds_tr[non_mask] == 0).sum()
            fp2 = (all_preds_tr[non_mask] == 1).sum()
            spec = tn / (tn + fp2) if (tn + fp2) > 0 else 0

            # Accuracy
            acc = (all_preds_tr == all_true_tr).mean()

            train_recall.append(rec)
            train_precision.append(prec)
            train_specificity.append(spec)
            train_accuracy.append(acc)
            train_epoch.append(epoch + 1)

            model.train()  # back to train mode
            print(f"  Epoch {epoch+1:>2}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | "
                  f"Recall: {rec:.3f} | Precision: {prec:.3f} | "
                  f"Specificity: {spec:.3f} | Acc: {acc:.3f}")

            # ---- LOG PER-EPOCH METRICS TO MLFLOW ----
            mlflow.log_metric("train_recall",      rec,      step=epoch)
            mlflow.log_metric("train_precision",   prec,     step=epoch)
            mlflow.log_metric("train_specificity", spec,     step=epoch)
            mlflow.log_metric("train_accuracy",    acc,      step=epoch)
            mlflow.log_metric("train_loss",        avg_loss, step=epoch)

        # ---- TEST EVALUATION (held-out patients, no leakage) ----
        model.eval()
        correct, total = 0, 0
        all_preds, all_true = [], []

        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                outputs = model(x)
                _, predicted = torch.max(outputs, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_true.extend(y.cpu().numpy())
                total += y.size(0)
                correct += (predicted == y).sum().item()

        all_preds = np.array(all_preds)
        all_true  = np.array(all_true)
        accuracy = 100 * correct / total

        print(f"  Test Accuracy: {accuracy:.2f}% ({correct}/{total})")

        seizure_mask = all_true == 1
        if seizure_mask.sum() > 0:
            tp = (all_preds[seizure_mask] == 1).sum()
            fn = (all_preds[seizure_mask] == 0).sum()
            sens = tp / (tp + fn)
            print(f"  Sensitivity (recall): {sens:.2f}  (caught {tp}/{tp+fn} seizures)")
        else:
            sens = 0.0

        non_mask = all_true == 0
        if non_mask.sum() > 0:
            tn = (all_preds[non_mask] == 0).sum()
            fp = (all_preds[non_mask] == 1).sum()
            spec = tn / (tn + fp)
            print(f"  Specificity:         {spec:.2f}  (rejected {tn}/{tn+fp} non-seizures)")
        else:
            spec = 0.0

        # ---- LOG TEST METRICS TO MLFLOW ----
        mlflow.log_metric("test_accuracy",    accuracy)
        mlflow.log_metric("test_recall",      sens)
        mlflow.log_metric("test_specificity", spec)

        # ---- SAVE MODEL ARTIFACT ----
        torch.save(model.state_dict(), f"model_weight_{seizure_weight:.0f}x.pt")
        mlflow.log_artifact(f"model_weight_{seizure_weight:.0f}x.pt")

        results.append({
            "weight": seizure_weight,
            "test_recall": sens,
            "test_specificity": spec,
            "test_accuracy": accuracy
        })

        # ---- THRESHOLD SWEEP ----
        model.eval()
        probs_all = []
        with torch.no_grad():
            for x, _ in test_loader:
                x = x.to(device)
                outputs = model(x)
                probs = torch.softmax(outputs, dim=1)[:, 1]
                probs_all.extend(probs.cpu().numpy())

        probs_all = np.array(probs_all)
        thresholds = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]

        print("\n" + f"  Threshold Tuning (weight={seizure_weight:.0f}x)")
        print(f"  {'Thresh':<8} {'Recall':<10} {'Spec':<10} {'FA/hr':<10}")
        print("  " + "-" * 38)

        for t in thresholds:
            preds = (probs_all >= t).astype(int)
            sez_mask = all_true == 1
            non_mask = all_true == 0
            if sez_mask.sum() > 0:
                tp = (preds[sez_mask] == 1).sum()
                fn = (preds[sez_mask] == 0).sum()
                rec = tp / (tp + fn)
            else:
                rec = 0.0
            if non_mask.sum() > 0:
                tn = (preds[non_mask] == 0).sum()
                fp = (preds[non_mask] == 1).sum()
                spec = tn / (tn + fp)
            else:
                spec = 0.0

            # Estimate false alarms per hour
            total_windows = len(all_true)
            window_sec = 7.0  # 1792 samples at 256 Hz
            stride_sec = window_sec * (1 - 0.3)
            windows_per_hour = 3600 / stride_sec
            fa_per_hour = fp / len(all_true) * windows_per_hour

            print(f"  {t:<8.2f} {rec:<10.3f} {spec:<10.3f} {fa_per_hour:<10.0f}")

        print()


  Epoch  1/13 | Loss: 0.1869 | Recall: 1.000 | Precision: 0.438 | Specificity: 0.000 | Acc: 0.438
  Epoch  2/13 | Loss: 0.2029 | Recall: 1.000 | Precision: 0.438 | Specificity: 0.000 | Acc: 0.438
  Epoch  3/13 | Loss: 0.1649 | Recall: 1.000 | Precision: 0.439 | Specificity: 0.006 | Acc: 0.441
  Epoch  4/13 | Loss: 0.1738 | Recall: 1.000 | Precision: 0.439 | Specificity: 0.006 | Acc: 0.441
  Epoch  5/13 | Loss: 0.1070 | Recall: 1.000 | Precision: 0.441 | Specificity: 0.012 | Acc: 0.444
  Epoch  6/13 | Loss: 0.1524 | Recall: 1.000 | Precision: 0.443 | Specificity: 0.022 | Acc: 0.450
  Epoch  7/13 | Loss: 0.2068 | Recall: 1.000 | Precision: 0.453 | Specificity: 0.062 | Acc: 0.472
  Epoch  8/13 | Loss: 0.1499 | Recall: 1.000 | Precision: 0.604 | Specificity: 0.490 | Acc: 0.713
  Epoch  9/13 | Loss: 0.1071 | Recall: 0.941 | Precision: 0.794 | Specificity: 0.810 | Acc: 0.867
  Epoch 10/13 | Loss: 0.1280 | Recall: 1.000 | Precision: 0.553 | Specificity: 0.370 | Acc: 0.646
  Epoch 11/13 | Loss

## Cell 9: Evaluate on Unseen Patients

Metrics: test set only — no leakage.

## Cell 8: Training Summary

Epoch-by-epoch comparison. Recall = % of seizures caught. Precision = % of seizure predictions that were correct. Specificity = % of non-seizures correctly ignored.

In [ ]:
import json, os

MODEL_DIR = "/kaggle/working/model"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save weights
torch.save(model.state_dict(), os.path.join(MODEL_DIR, "eegcnn1d_weights.pth"))

# Save config
config = {
    "model": "EEGCNN1D",
    "n_channels": int(X_train.shape[1]),
    "n_time_samples": int(X_train.shape[2]),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "random_seed": RANDOM_SEED,
    "data_dir": DATA_DIR,
}
with open(os.path.join(MODEL_DIR, "model_config.json"), "w") as f:
    json.dump(config, f, indent=2)
    f.flush()

# Save test metrics — convert numpy types to Python native
if results:
    best = max(results, key=lambda r: r["test_recall"])
    metrics = {
        "test_accuracy": float(best["test_accuracy"]),
        "sensitivity": float(best["test_recall"]),
        "specificity": float(best["test_specificity"]),
        "seizure_weight": float(best["weight"]),
    }
    with open(os.path.join(MODEL_DIR, "test_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
        f.flush()

# Verify
print("Saved to /kaggle/working/model/")
for fn in sorted(os.listdir(MODEL_DIR)):
    sz = os.path.getsize(os.path.join(MODEL_DIR, fn))
    if sz < 1000:
        print(f"  {fn} ({sz} B)")
    else:
        print(f"  {fn} ({sz/1e3:.0f} KB)")

Saved to /kaggle/working/model/
  eegcnn1d_weights.pth (697 KB)
  model_config.json (230 B)
  test_metrics.json (139 B)


In [ ]:
print(f"{'Epoch':<6} {'Recall':<8} {'Precision':<10} {'Specificity':<12} {'Accuracy':<9}")
print("-" * 50)
for i in range(len(train_epoch)):
    print(f"{train_epoch[i]:<6}"
          f"{train_recall[i]:<8.3f}"
          f"{train_precision[i]:<10.3f}"
          f"{train_specificity[i]:<12.3f}"
          f"{train_accuracy[i]:<9.3f}")

Epoch  Recall   Precision  Specificity  Accuracy 
--------------------------------------------------
1     1.000   0.438     0.000       0.438    
2     1.000   0.438     0.000       0.438    
3     1.000   0.439     0.006       0.441    
4     1.000   0.439     0.006       0.441    
5     1.000   0.441     0.012       0.444    
6     1.000   0.443     0.022       0.450    
7     1.000   0.453     0.062       0.472    
8     1.000   0.604     0.490       0.713    
9     0.941   0.794     0.810       0.867    
10    1.000   0.553     0.370       0.646    
11    0.997   0.692     0.654       0.804    
12    0.990   0.786     0.790       0.877    
13    0.997   0.817     0.826       0.901    


In [ ]:
!zip -r model.zip /kaggle/working/model/
from IPython.display import FileLink
FileLink("model.zip")

  adding: kaggle/working/model/ (stored 0%)
  adding: kaggle/working/model/model_config.json (deflated 27%)
  adding: kaggle/working/model/test_metrics.json (deflated 23%)
  adding: kaggle/working/model/eegcnn1d_weights.pth (deflated 8%)


/kaggle/working/model.zip